# Visualisation : chaque variable par rapport au loyer

Données : `data_cleaning/annonces.csv` (sortie de `data_cleaning.py`).
Les figures sont aussi enregistrées dans le dossier `figures/` pour le rapport.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLEU, BLEU_CLAIR, BLEU_FONCE = "#2a78d6", "#cde2fb", "#184f95"
ENCRE, ENCRE_2, GRILLE = "#0b0b0b", "#52514e", "#e5e4e0"

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": GRILLE, "axes.labelcolor": ENCRE_2,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRILLE, "grid.linewidth": 1,
    "axes.axisbelow": True, "xtick.color": ENCRE_2, "ytick.color": ENCRE_2,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "font.size": 10,
})

os.makedirs("figures", exist_ok=True)

## 1. Chargement des données

In [ ]:
df = pd.read_csv("data_cleaning/annonces.csv", sep=";").drop_duplicates()
CIBLE = "Loyer_TTC"

# Le stationnement est déjà regroupé en 4 catégories dans data_cleaning.py
df["Quartiers"] = df["Quartiers"].str.replace("Quartiers ", "")

print(len(df), "annonces")
df.head()

## 2. Fonctions de tracé

- `nuage` : variable numérique → nuage de points + droite de régression, avec la corrélation r dans le titre.
- `boites` : variable catégorielle → une boîte par modalité (médiane, quartiles), avec les annonces en points gris et l'effectif n.

In [ ]:
def nuage(ax, col, unite=""):
    x, y = df[col], df[CIBLE]
    ax.scatter(x, y, s=22, color=BLEU, alpha=0.55, edgecolor="#fcfcfb", linewidth=0.6)
    a, b = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, a * xs + b, color=ENCRE, linewidth=2)
    ax.set_title(f"{col}  (r = {x.corr(y):.2f})")
    ax.set_xlabel(f"{col} {unite}".strip())
    ax.set_ylabel("Loyer (€)")
    ax.grid(axis="x", color=GRILLE)


def boites(ax, col, ordre=None, etiquettes=None, titre=None):
    if ordre is None:  # par défaut : modalités triées par loyer médian
        ordre = df.groupby(col)[CIBLE].median().sort_values().index.tolist()
    groupes = [df.loc[df[col] == m, CIBLE] for m in ordre]
    ax.boxplot(groupes, widths=0.55, patch_artist=True, showfliers=False,
               medianprops=dict(color=BLEU_FONCE, linewidth=2),
               boxprops=dict(facecolor=BLEU_CLAIR, edgecolor=BLEU, linewidth=1),
               whiskerprops=dict(color=BLEU, linewidth=1),
               capprops=dict(color=BLEU, linewidth=1))
    rng = np.random.default_rng(0)
    for k, g in enumerate(groupes, start=1):
        ax.scatter(k + rng.uniform(-0.15, 0.15, len(g)), g, s=8, color=ENCRE_2, alpha=0.35, linewidth=0)
    noms = etiquettes or [str(m) for m in ordre]
    ax.set_xticks(range(1, len(ordre) + 1), [f"{n}\n(n={len(g)})" for n, g in zip(noms, groupes)])
    ax.set_title(titre or col)
    ax.set_ylabel("Loyer (€)")

## 3. Variables numériques

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, (col, unite) in zip(axes.flat, [("Surface", "(m²)"), ("Charges", "(€)"),
                                        ("kWh", "(kWh/m²/an)"), ("kgCO2", "(kgCO2/m²/an)")]):
    nuage(ax, col, unite)
fig.suptitle("Variables numériques et loyer", x=0.01, ha="left", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig("figures/numeriques_loyer.png", dpi=150)
plt.show()

## 4. Variables catégorielles

In [ ]:
pieces = sorted(df["Pièces"].unique())

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
boites(axes[0, 0], "Pièces", ordre=pieces, etiquettes=[str(int(p)) for p in pieces], titre="Nombre de pièces")
boites(axes[0, 1], "label_eco", ordre=["B+", "C", "D", "E", "F"], titre="Classe énergétique (DPE)")
boites(axes[1, 0], "Quartiers", titre="Quartier")
boites(axes[1, 1], "station", titre="Stationnement")
fig.suptitle("Variables catégorielles et loyer", x=0.01, ha="left", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig("figures/categorielles_loyer.png", dpi=150)
plt.show()

## 5. Résumé : corrélation de chaque variable avec le loyer

Corrélation de Spearman (fondée sur les rangs : moins sensible aux valeurs extrêmes que Pearson).

In [ ]:
num = ["Surface", "Pièces", "Charges", "kWh", "kgCO2"]
dpe = df["label_eco"].map({"B+": 1, "C": 2, "D": 3, "E": 4, "F": 5}).rename("label_eco (B+ → F)")
corr = (pd.concat([df[num], dpe], axis=1)
        .corrwith(df[CIBLE], method="spearman")
        .sort_values(key=abs, ascending=False)
        .round(2)
        .rename("corrélation avec le loyer"))
corr.to_frame()